In [1]:
import numpy as np

# AWG Initialisation

In [2]:
#Jupyter cell 1 — imports and connection
from rigol_dg1022 import RigolDG1022

RESOURCE = "USB0::0x1AB1::0x0642::DG1ZA231701902::INSTR"  # update if your VISA address differs
gen = RigolDG1022(RESOURCE)  # auto_open=True by default
print("Connected to:", gen.idn)

Connected to: Rigol Technologies,DG1022Z,DG1ZA231701902,03.01.12


## Set Wavefrom

In [3]:
# Sine wave on CH1, 2 Vpp, 0 V offset
freq=int(1e6)

Vpi=5.97
coeff=.8
Vpp_Ch1=round(Vpi*coeff,3)
#Set waveform on CH1
gen.set_waveform(ch=1, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch1, offset_v=0.0)

#Set phase of CH1
gen.set_phase_deg(ch=1, phase_deg=0) 

#Set ch1 impedance to 50 Ohms or INF
gen.set_load(ch=1, load='50')




# PhasemeterClient — quick start

In [4]:
from phasemeter_client_fixed import PhasemeterClient


# Fill in the target of your Moku (e.g., '192.168.1.100' or link-local IPv6)
TARGET = '[fe80::7269:79ff:feb7:d15%6]'  # <-- replace if needed
client = PhasemeterClient(
    target=TARGET,
    phase_units='cycles',   # or 'deg' depending on API
    wrap_output=True,
    input_range='1Vpp',
    impedance='50Ohm',
    coupling='DC',
    pll_bandwidth='1kHz',
    poll_sec=10,
)
client.load(f0_hz=1e6)  # optional initial lock

## Change device settings on the fly

In [5]:
# Change front-end input range for channel 1, then both
client.set_frontend(1, input_range='1Vpp')
client.set_frontend(2, input_range='1Vpp')
# Change PLL bandwidth for both channels
client.set_bandwidth_all('1kHz')
# Move center frequency (both channels track the same f)
client.set_frequency_all(1e6)

# Laser Setup

In [6]:
import TLX_5

import time

laser=TLX_5.TRL_5()

port="COM4"

laser.connect(port)

Serial<id=0x1c9f83a9c90, open=True>(port='COM4', baudrate=115200, bytesize=8, parity='N', stopbits=1, timeout=1.0, xonxoff=False, rtscts=False, dsrdtr=False)

## Set Laser Wavelength (nm)

In [19]:
wl="1550.0"

laser.change_WL(wl)  # in nm

laser.laser_ON()


'1'

# Test 

In [32]:
# Set Sine wave frequency
f=20e6

freq=int(f)
gen.set_waveform(ch=1, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch1, offset_v=0.0)


In [33]:
# Move center frequency (both channels track the same f)
client.set_frequency_all(f)


In [22]:
#Turn Ch1 ouput ON
gen.output_on(1)

time.sleep(5)

dphi_deg, frame = client.read_phase_difference_deg()
print('Δφ [deg]:', dphi_deg)
print('f1 [Hz]:', frame['ch1']['frequency'], 'f2 [Hz]:', frame['ch2']['frequency'])

#laser.laser_OFF()

gen.output_off(1)

Δφ [deg]: 6.621730327606201
f1 [Hz]: 20000011.627207216 f2 [Hz]: 20000011.625690207


In [30]:
def mesurement_avg(n_av:int):
    ph_diff=[]
    f1=[]
    f2=[]
    time.sleep(5)
    i=0
    while i<n_av:
       dphi_deg, frame = client.read_phase_difference_deg()
       ph_diff.append(dphi_deg)
       f1.append( frame['ch1']['frequency'])
       f2.append( frame['ch2']['frequency'])
       i+=1
    f1_avg=np.mean(f1)
    f2_avg=np.mean(f2)
    ph_diff_avg=np.mean(ph_diff)
    ph_diff_std=np.std(ph_diff)
    return f1_avg,f2_avg,ph_diff_avg,ph_diff_std
        

In [36]:
gen.output_on(1)

results=mesurement_avg(40)

gen.output_off(1)

print(f'f1_avg={results[0]} Hz; f2_avg={results[1]} Hz; ph_diff_avg={results[2]} ± {results[3]}\n')




f1_avg=20000011.720607705 Hz; f2_avg=20000011.720561516 Hz; ph_diff_avg=6.628623872995377 ± 0.0018804488800334175



In [37]:
def f_sweep(n_av:int, freq):
    size=len(freq)
    i=0
    f1=[]
    f2=[]
    phase=[]
    gen.output_on(1)
    while i<size:
        f=int(freq[i])
        gen.set_waveform(ch=1, wave="SIN", freq_hz=f, ampl_vpp=Vpp_Ch1, offset_v=0.0)
        client.set_frequency_all(f)
        results=mesurement_avg(20)
        print(f'f1_avg={results[0]} Hz; f2_avg={results[1]} Hz; ph_diff_avg={results[2]} ± {results[3]}\n')
        f1.append(results[0])
        f2.append(results[1])
        phase.append(results[2])
        i+=1
    gen.output_off(1)
        
    return(f1,f2,phase)
        
        
        
        


In [43]:
freqs=np.linspace(6e6,22e6,17)
n=30

res1=f_sweep(n,freqs)

f1_avg=6000003.504799077 Hz; f2_avg=6000003.504794193 Hz; ph_diff_avg=7.149829119443893 ± 0.0005903124036206602

f1_avg=7000004.087916876 Hz; f2_avg=7000004.087869361 Hz; ph_diff_avg=7.253714025020599 ± 0.0008274918673127957

f1_avg=8000004.6745700715 Hz; f2_avg=8000004.674468376 Hz; ph_diff_avg=7.306745320558548 ± 0.0008103920250068034

f1_avg=9000005.245060638 Hz; f2_avg=9000005.245123701 Hz; ph_diff_avg=7.322652965784073 ± 0.0007941757287633633

f1_avg=10000005.846777605 Hz; f2_avg=10000005.846857097 Hz; ph_diff_avg=7.314267843961716 ± 0.0014853222777805697

f1_avg=11000006.430291977 Hz; f2_avg=11000006.4301075 Hz; ph_diff_avg=7.285494446754456 ± 0.0008823699649620936

f1_avg=12000007.008628534 Hz; f2_avg=12000007.008642033 Hz; ph_diff_avg=7.2428363263607025 ± 0.001307825395005724

f1_avg=13000007.560510695 Hz; f2_avg=13000007.560652805 Hz; ph_diff_avg=7.184951037168503 ± 0.0010989655171302203

f1_avg=14000008.175245518 Hz; f2_avg=14000008.175432567 Hz; ph_diff_avg=7.121496677398682

In [46]:
res2=f_sweep(n,freqs)

f1_avg=6000003.496940654 Hz; f2_avg=6000003.497152841 Hz; ph_diff_avg=7.14369598031044 ± 0.0010171613833847679

f1_avg=7000004.08549224 Hz; f2_avg=7000004.085310785 Hz; ph_diff_avg=7.2509154081344604 ± 0.0010638756790878443

f1_avg=8000004.676778528 Hz; f2_avg=8000004.676759253 Hz; ph_diff_avg=7.307497680187225 ± 0.0006551008760570984

f1_avg=9000005.262084354 Hz; f2_avg=9000005.262003973 Hz; ph_diff_avg=7.324519246816635 ± 0.0009417407737223702

f1_avg=10000005.842116889 Hz; f2_avg=10000005.841981085 Hz; ph_diff_avg=7.31483781337738 ± 0.0013850283275347278

f1_avg=11000006.42388199 Hz; f2_avg=11000006.423908103 Hz; ph_diff_avg=7.285412639379501 ± 0.0013783895614457038

f1_avg=12000006.999214375 Hz; f2_avg=12000006.999128666 Hz; ph_diff_avg=7.242493271827698 ± 0.0009173725762920592

f1_avg=13000007.582893325 Hz; f2_avg=13000007.58280433 Hz; ph_diff_avg=7.184552192687988 ± 0.0010726923474068317

f1_avg=14000008.179263458 Hz; f2_avg=14000008.17911149 Hz; ph_diff_avg=7.121921539306641 ± 0

In [47]:

i=0
while i<len(freqs):
    print(f'Calibration={res2[2][i]-res1[2][i]}\n')
    i+=1

Calibration=-0.006133139133453369

Calibration=-0.002798616886138916

Calibration=0.0007523596286773682

Calibration=0.0018662810325622559

Calibration=0.0005699694156646729

Calibration=-8.180737495422363e-05

Calibration=-0.00034305453300476074

Calibration=-0.00039884448051452637

Calibration=0.0004248619079589844

Calibration=0.0009385049343109131

Calibration=4.774332046508789e-05

Calibration=8.100271224975586e-05

Calibration=0.0022031664848327637

Calibration=0.0032602250576019287

Calibration=6.946921348571777e-05

Calibration=-0.0012246966361999512

Calibration=-0.00024515390396118164



In [48]:
laser.laser_OFF()

'1'